# siann's Triton probe, against cuVS

Run on Colab with a T4 GPU (Runtime, Change runtime type, T4 GPU), then Runtime, Run all: it finishes within 5
minutes. Colab comes with PyTorch, Triton and cuVS, so almost nothing is installed. siann's exact L1 kernels in
Triton (the selection merged inside the kernel, and the earlier 2D-tile kernel) are timed against cuVS's brute
force on 200,000 Gaussian vectors, with the memory each takes; the benchmark stops itself at what is left of the
5 minutes. Copy the output of the last step back.

In [ ]:
# Step 1 of 4: Which GPU (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 1 of 4: Which GPU, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(f"step 1 of 4: Which GPU: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 2 of 4, Small installs", flush=True)

In [ ]:
# Step 2 of 4: Small installs (about 1 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 2 of 4: Small installs, about 1 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
# small packages for the memory figures; cuVS only where missing (Colab comes with it), for this CUDA
import subprocess
import sys

import torch

cuda = torch.version.cuda.split(".")[0]
print("PyTorch", torch.__version__, "for CUDA", torch.version.cuda)


def works(stmt):
    return subprocess.run([sys.executable, "-c", stmt], check=False, capture_output=True).returncode == 0


def install(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg, "--extra-index-url=https://pypi.nvidia.com"],
                       check=False, capture_output=True, text=True)
    print(pkg, "installed" if r.returncode == 0 else "not installed: " + (r.stderr.strip().splitlines() or ["?"])[-1])
    return r.returncode == 0


for pkg in ("nvidia-ml-py", "psutil"):
    install(pkg)
if not works("from cuvs.neighbors import brute_force"):
    print("cuVS missing: installing it takes several minutes (Colab comes with it)")
    install(f"libcuvs-cu{cuda}")
    install(f"cuvs-cu{cuda}")
print("cuVS", "works" if works("from cuvs.neighbors import brute_force") else "does not work: the benchmark skips it")
print(f"step 2 of 4: Small installs: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 3 of 4, Write triton_probe.py", flush=True)

In [ ]:
# Step 3 of 4: Write triton_probe.py (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 3 of 4: Write triton_probe.py, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
source = "\"\"\"A probe of siann's GPU kernels in Triton (NVIDIA and AMD GPUs): exact L1 search with the selection fused\ninto the distance kernel, siann's two-stage design. Stage 1: each program takes BLOCK_Q queries and BLOCK_N\nbase vectors, accumulates their L1 distances BLOCK_D coordinates at a time, and keeps each query's best k of\nthe BLOCK_N (one 64-bit key each: the distance's bits, then the vector's index, so that ties order by index),\nby sorting them or by k passes of a minimum. Triton's auto-tuner picks block sizes, selection and warps. Stage 2: torch.topk over each query's survivors, k per block of the base.\n\n--check: the neighbors must equal exact search's, ties included; on a GPU, or on any machine through Triton's\ninterpreter (TRITON_INTERPRET=1, on the CPU). --bench, on an NVIDIA GPU: queries per second against torch.cdist with\ntorch.topk, FAISS's GPU brute force and cuVS's brute force, all by L1, on Gaussian vectors, and the memory each\ntakes: PyTorch's peak (exact, but blind to FAISS's and cuVS's own allocators), the device's peak above the level\nbefore the call (NVIDIA's management library, polled every 10 ms: short spikes may be missed), the device memory\neach keeps between calls (FAISS's scratch memory, cuVS's index), and the host process's peak.\n\n    TRITON_INTERPRET=1 python triton_probe.py --check          # anywhere, on the CPU\n    python triton_probe.py --bench                              # on a CUDA GPU (Colab, Kaggle)\n\"\"\"\nimport os\n\n# FAISS's OpenMP threads, once FAISS has run, spin while idle and slowed siann's small batches on 2-4 core\n# machines (Colab, Kaggle): idle threads sleep instead. FAISS is also first called after siann is timed.\nos.environ.setdefault(\"OMP_WAIT_POLICY\", \"PASSIVE\")\n\nimport argparse\nimport gc\nimport threading\nimport time\n\nimport numpy as np\nimport torch\nimport triton\nimport triton.language as tl\n\nMAX_KEY = 0x7FFFFFFFFFFFFFFF\nBEST_2D = {\"BLOCK_Q\": 32, \"BLOCK_N\": 128, \"SORT\": True, \"num_warps\": 4}   # the T4's fastest of the 24 (Colab)\n\n\nclass Deadline:\n    \"\"\"A hard limit on the whole run: work checks it before each step, and once it has passed, the step is\n    skipped with a line saying so (the results so far stand).\"\"\"\n\n    def __init__(self, seconds: float):\n        self.seconds, self.end = seconds, time.perf_counter() + seconds\n\n    def over(self, what: str) -> bool:\n        if time.perf_counter() < self.end:\n            return False\n        print(f\"  out of time (the {self.seconds:.0f} s budget): {what} skipped\", flush=True)\n        return True\n\n\nDEADLINE = Deadline(float(\"inf\"))\n\n\n@triton.jit\ndef _l1_block_topk(q_ptr, x_ptr, out_ptr, nq, n, d, stride,\n                   BLOCK_Q: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_D: tl.constexpr, K: tl.constexpr,\n                   SORT: tl.constexpr):\n    pq = tl.program_id(0)\n    pn = tl.program_id(1)\n    rq = pq * BLOCK_Q + tl.arange(0, BLOCK_Q)\n    rn = pn * BLOCK_N + tl.arange(0, BLOCK_N)\n    acc = tl.zeros((BLOCK_Q, BLOCK_N), dtype=tl.float32)\n    for d0 in range(0, d, BLOCK_D):\n        rd = d0 + tl.arange(0, BLOCK_D)\n        q = tl.load(q_ptr + rq[:, None] * d + rd[None, :], mask=(rq[:, None] < nq) & (rd[None, :] < d), other=0.0)\n        x = tl.load(x_ptr + rn[:, None] * d + rd[None, :], mask=(rn[:, None] < n) & (rd[None, :] < d), other=0.0)\n        acc += tl.sum(tl.abs(q[:, None, :] - x[None, :, :]), axis=2)\n    _keep_best(acc, rq, rn, pn, out_ptr, nq, n, stride, BLOCK_N, K, SORT)\n\n\n@triton.jit\ndef _l1_tile_topk(qt_ptr, xt_ptr, out_ptr, nq, n, d, stride,\n                  BLOCK_Q: tl.constexpr, BLOCK_N: tl.constexpr, K: tl.constexpr, SORT: tl.constexpr):\n    \"\"\"The same search with the queries and base vectors stored transposed (one dimension at a time): each step\n    reads one dimension of BLOCK_Q queries and of BLOCK_N base vectors, both contiguous, and adds their\n    differences into a 2D tile of running distances, as a matrix multiply's rank-1 update does.\"\"\"\n    pq = tl.program_id(0)\n    pn = tl.program_id(1)\n    rq = pq * BLOCK_Q + tl.arange(0, BLOCK_Q)\n    rn = pn * BLOCK_N + tl.arange(0, BLOCK_N)\n    acc = tl.zeros((BLOCK_Q, BLOCK_N), dtype=tl.float32)\n    for j in range(d):\n        qj = tl.load(qt_ptr + j * nq + rq, mask=rq < nq, other=0.0)\n        xj = tl.load(xt_ptr + j * n + rn, mask=rn < n, other=0.0)\n        acc += tl.abs(qj[:, None] - xj[None, :])\n    _keep_best(acc, rq, rn, pn, out_ptr, nq, n, stride, BLOCK_N, K, SORT)\n\n\n@triton.jit\ndef _keep_best(acc, rq, rn, pn, out_ptr, nq, n, stride, BLOCK_N: tl.constexpr, K: tl.constexpr,\n               SORT: tl.constexpr):\n    \"\"\"Each query's best K of the block, written to its slots of the survivors.\"\"\"\n    # one sortable key per candidate: the (non-negative) distance's bits above, the index below\n    bits = acc.to(tl.int32, bitcast=True).to(tl.int64)\n    key = (bits << 32) | rn[None, :].to(tl.int64)\n    key = tl.where(rn[None, :] < n, key, 0x7FFFFFFFFFFFFFFF)\n    if SORT:                                    # the whole block sorted, its first K kept\n        key = tl.sort(key, dim=1)\n        cols = tl.arange(0, BLOCK_N)\n        out = out_ptr + rq[:, None] * stride + pn * K + cols[None, :]\n        tl.store(out, key, mask=(rq[:, None] < nq) & (cols[None, :] < K))\n    else:                                       # K passes, each taking the smallest key (keys are unique)\n        for j in tl.static_range(K):\n            m = tl.min(key, axis=1)\n            tl.store(out_ptr + rq * stride + pn * K + j, m, mask=rq < nq)\n            key = tl.where(key == m[:, None], 0x7FFFFFFFFFFFFFFF, key)\n\n\n@triton.jit\ndef _l1_merge_topk(qt_ptr, xt_ptr, out_ptr, nq, n, d, per_split,\n                   BLOCK_Q: tl.constexpr, BLOCK_N: tl.constexpr, K: tl.constexpr, KP: tl.constexpr):\n    \"\"\"The in-kernel merge: each program takes BLOCK_Q queries and one stretch of the base (per_split vectors),\n    scans it BLOCK_N vectors at a time as the 2D-tile kernel does, and keeps its queries' best K in registers\n    (KP lanes, K of them used, in no order). Each block's candidates no nearer than their query's current K-th\n    are dropped, and then as many passes run as the busiest query has candidates left (few, once the first\n    blocks are in), each replacing a query's worst kept key by its best new one. Only the K keys are written,\n    so stage 2 sorts K per stretch, not K per block.\"\"\"\n    pq = tl.program_id(0)\n    ps = tl.program_id(1)\n    nsplit = tl.num_programs(1)\n    rq = pq * BLOCK_Q + tl.arange(0, BLOCK_Q)\n    lane = tl.arange(0, KP)\n    # the best K so far, in no order: MAX where not yet filled; -1 in the KP - K unused lanes (never the worst)\n    run = tl.where(lane[None, :] < K, tl.full((BLOCK_Q, KP), 0x7FFFFFFFFFFFFFFF, tl.int64), -1)\n    start = ps * per_split\n    stop = tl.minimum(start + per_split, n)\n    for s0 in range(start, stop, BLOCK_N):\n        rn = s0 + tl.arange(0, BLOCK_N)\n        acc = tl.zeros((BLOCK_Q, BLOCK_N), dtype=tl.float32)\n        for j in range(d):\n            qj = tl.load(qt_ptr + j * nq + rq, mask=rq < nq, other=0.0)\n            xj = tl.load(xt_ptr + j * n + rn, mask=rn < stop, other=0.0)\n            acc += tl.abs(qj[:, None] - xj[None, :])\n        bits = acc.to(tl.int32, bitcast=True).to(tl.int64)\n        key = (bits << 32) | rn[None, :].to(tl.int64)\n        key = tl.where(rn[None, :] < stop, key, 0x7FFFFFFFFFFFFFFF)\n        worst = tl.max(run, axis=1)                                   # each query's K-th best so far\n        key = tl.where(key < worst[:, None], key, 0x7FFFFFFFFFFFFFFF)  # only candidates that would get in\n        passes = tl.minimum(tl.max(tl.sum((key < 0x7FFFFFFFFFFFFFFF).to(tl.int32), axis=1), axis=0), K)\n        for _ in range(passes):                                        # keys unique: each pass moves one in\n            m = tl.min(key, axis=1)\n            worst = tl.max(run, axis=1)\n            take = m < worst\n            slot = tl.min(tl.where(run == worst[:, None], lane[None, :], KP), axis=1)   # one slot: empty ones tie\n            run = tl.where((lane[None, :] == slot[:, None]) & take[:, None], m[:, None], run)\n            key = tl.where(key == m[:, None], 0x7FFFFFFFFFFFFFFF, key)\n    out = out_ptr + rq[:, None] * (nsplit * K) + ps * K + lane[None, :]\n    tl.store(out, run, mask=(rq[:, None] < nq) & (lane[None, :] < K))\n\n\nMERGE_DEFAULT = {\"BLOCK_Q\": 32, \"BLOCK_N\": 128, \"num_warps\": 4}\nCONFIGS_MERGE = [{\"BLOCK_Q\": bq, \"BLOCK_N\": bn, \"num_warps\": w}\n                 for bq, bn in ((32, 128), (64, 128), (16, 128), (32, 64)) for w in (4, 8)]\n\n\ndef l1_knn_merge(xt: torch.Tensor, q: torch.Tensor, k: int, config: dict | None = None,\n                 programs: int | None = None) -> torch.Tensor:\n    \"\"\"Each query's k nearest by L1 through the in-kernel merge (xt: the base transposed, d x n, contiguous),\n    int64 indices, nearest first, ties by index. The base is split into stretches so that about ``programs``\n    programs run (by default eight per streaming multiprocessor): many for small batches, few for large ones.\"\"\"\n    d, n = xt.shape\n    nq = q.shape[0]\n    cfg = dict(config or MERGE_DEFAULT)\n    warps = cfg.pop(\"num_warps\", 4)\n    bq, bn = cfg[\"BLOCK_Q\"], cfg[\"BLOCK_N\"]\n    if programs is None:\n        programs = 8 * (torch.cuda.get_device_properties(xt.device).multi_processor_count\n                        if xt.device.type == \"cuda\" else 16)\n    qblocks = triton.cdiv(nq, bq)\n    nblocks = triton.cdiv(n, bn)\n    split = max(1, min(nblocks, triton.cdiv(programs, qblocks)))\n    per = triton.cdiv(nblocks, split) * bn                    # whole blocks per stretch\n    split = triton.cdiv(n, per)\n    kp = 1 << (k - 1).bit_length()\n    out = torch.empty((nq, split * k), dtype=torch.int64, device=xt.device)\n    _l1_merge_topk[(qblocks, split)](q.T.contiguous(), xt, out, nq, n, d, per, BLOCK_Q=bq, BLOCK_N=bn, K=k, KP=kp,\n                                     num_warps=warps)\n    return torch.topk(out, k, dim=1, largest=False, sorted=True).values & 0xFFFFFFFF\n\n\n# Tiles of about 16,000 values (BLOCK_Q x BLOCK_N x BLOCK_D), larger ones spilling out of the registers; both\n# selections; 4 or 8 warps. Triton's auto-tuner times each once per shape and keeps the fastest.\nTILES = ((16, 128, 8), (32, 128, 4), (64, 64, 4), (32, 64, 8), (16, 256, 4), (64, 128, 2))\nCONFIGS = [triton.Config({\"BLOCK_Q\": bq, \"BLOCK_N\": bn, \"BLOCK_D\": bd, \"SORT\": sort}, num_warps=w)\n           for bq, bn, bd in TILES for sort in (True, False) for w in (4, 8)]\n# the 2D-tile kernel: no third dimension in registers, so larger tiles\nTILES_2D = ((32, 128), (64, 128), (128, 128), (32, 256), (64, 256), (128, 64))\nCONFIGS_2D = [triton.Config({\"BLOCK_Q\": bq, \"BLOCK_N\": bn, \"SORT\": sort}, num_warps=w)\n              for bq, bn in TILES_2D for sort in (True, False) for w in (4, 8)]\nMIN_BLOCK_N = min([bn for _, bn, _ in TILES] + [bn for _, bn in TILES_2D])\n# each trial of the auto-tuner restores the survivors: configurations with other block sizes write other slots\n_l1_tuned = triton.autotune(configs=CONFIGS, key=[\"n\", \"d\", \"K\"], restore_value=[\"out_ptr\"])(_l1_block_topk)\n_tiles_tuned = triton.autotune(configs=CONFIGS_2D, key=[\"n\", \"d\", \"K\"], restore_value=[\"out_ptr\"])(_l1_tile_topk)\n\n\nSURVIVOR_BYTES = 1 << 30   # stage 1's output per call at most: larger batches go in chunks of queries\n\n\ndef l1_knn(x: torch.Tensor, q: torch.Tensor, k: int, config: dict | None = None, xt: torch.Tensor | None = None):\n    \"\"\"Each query's k nearest rows of x by L1 (int64 indices, nearest first, ties by index): with the block sizes\n    and selection of ``config`` ({\"BLOCK_Q\", \"BLOCK_N\", \"BLOCK_D\", \"SORT\"}, plus \"num_warps\"), or else the\n    fastest Triton's auto-tuner finds. With ``xt`` (x transposed, d x n, contiguous), the 2D-tile kernel.\"\"\"\n    nq, d = q.shape\n    n = x.shape[0]\n    tiles = xt is not None\n    kernel, tuned = (_l1_tile_topk, _tiles_tuned) if tiles else (_l1_block_topk, _l1_tuned)\n    base = xt if tiles else x\n    width = triton.cdiv(n, MIN_BLOCK_N) * k         # room for the smallest blocks' survivors; unused slots stay MAX\n    step = max(64, SURVIVOR_BYTES // (width * 8) // 64 * 64)\n    out = []\n    for s in range(0, nq, step):\n        qs = q[s:s + step]\n        survivors = torch.full((len(qs), width), MAX_KEY, dtype=torch.int64, device=x.device)\n        m = len(qs)\n        qa = qs.T.contiguous() if tiles else qs\n        if config is None:\n            def grid(meta, m=m):\n                return (triton.cdiv(m, meta[\"BLOCK_Q\"]), triton.cdiv(n, meta[\"BLOCK_N\"]))\n            tuned[grid](qa, base, survivors, m, n, d, width, K=k)\n        else:\n            cfg = dict(config)\n            warps = cfg.pop(\"num_warps\", 4)\n            grid = (triton.cdiv(m, cfg[\"BLOCK_Q\"]), triton.cdiv(n, cfg[\"BLOCK_N\"]))\n            kernel[grid](qa, base, survivors, m, n, d, width, K=k, num_warps=warps, **cfg)\n        out.append(torch.topk(survivors, k, dim=1, largest=False, sorted=True).values & 0xFFFFFFFF)\n    return torch.cat(out)\n\n\ndef check(both: bool = False) -> None:\n    \"\"\"Against exact search by NumPy, ties broken by index, on sizes that leave partial blocks: on the GPU where\n    there is one (the compiled kernel), otherwise on the CPU through Triton's interpreter.\"\"\"\n    dev = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"  on {dev}\", flush=True)\n    rng = np.random.default_rng(0)\n    for n, d, nq, k in ((1000, 24, 37, 10), (777, 13, 5, 3), (300, 8, 20, 10)):\n        x = rng.integers(0, 4, (n, d)).astype(np.float32)          # small integers: many ties\n        q = rng.integers(0, 4, (nq, d)).astype(np.float32)\n        dist = np.abs(q[:, None, :] - x[None, :, :]).sum(axis=2)\n        truth = np.lexsort((np.broadcast_to(np.arange(n), dist.shape), dist), axis=1)[:, :k]\n        xd, qd = torch.from_numpy(x).to(dev), torch.from_numpy(q).to(dev)\n        xtd = xd.T.contiguous()\n        same = []\n        for cfg in CONFIGS_MERGE:                                  # the in-kernel merge, few and many stretches\n            for programs in (1, 7, 64):\n                same.append(np.array_equal(l1_knn_merge(xtd, qd, k, cfg, programs).cpu().numpy(), truth))\n                print(\".\" if same[-1] else \"x\", end=\"\", flush=True)            # one mark per case\n        if both:\n            for c in CONFIGS_2D:\n                cfg = {**c.kwargs, \"num_warps\": c.num_warps}\n                same.append(np.array_equal(l1_knn(xd, qd, k, cfg, xt=xtd).cpu().numpy(), truth))\n                print(\".\" if same[-1] else \"x\", end=\"\", flush=True)\n            for c in CONFIGS:\n                cfg = {**c.kwargs, \"num_warps\": c.num_warps}\n                same.append(np.array_equal(l1_knn(xd, qd, k, cfg).cpu().numpy(), truth))\n                print(\".\" if same[-1] else \"x\", end=\"\", flush=True)\n        print(f\"\\n  n {n}, d {d}, {nq} queries, k {k}: identical to exact search, ties included, in \"\n              f\"{sum(same)} of {len(same)} cases\")\n        assert all(same)\n\n\nclass Peaks:\n    \"\"\"Peak memory during a block, in MB above the level at its start: the device's (all allocators, polled\n    through NVIDIA's management library) and the host process's; None where a library is missing.\"\"\"\n\n    def __init__(self, interval: float = 0.01):\n        self.interval = interval\n        try:\n            import pynvml\n            pynvml.nvmlInit()\n            self._nvml = pynvml\n            self._handle = pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())\n        except Exception:  # noqa: BLE001 (no NVML: device peaks unknown)\n            self._nvml = None\n        try:\n            import psutil\n            self._proc = psutil.Process()\n        except Exception:  # noqa: BLE001\n            self._proc = None\n\n    def device_used(self):\n        return None if self._nvml is None else self._nvml.nvmlDeviceGetMemoryInfo(self._handle).used / 2**20\n\n    def _host(self):\n        return None if self._proc is None else self._proc.memory_info().rss / 2**20\n\n    def __enter__(self):\n        self.d0, self.h0 = self.device_used(), self._host()\n        self.dmax, self.hmax = self.d0, self.h0\n        self._stop = threading.Event()\n\n        def poll():\n            while not self._stop.is_set():\n                d, h = self.device_used(), self._host()\n                if d is not None:\n                    self.dmax = max(self.dmax, d)\n                if h is not None:\n                    self.hmax = max(self.hmax, h)\n                time.sleep(self.interval)\n        self._thread = threading.Thread(target=poll, daemon=True)\n        self._thread.start()\n        return self\n\n    def __exit__(self, *exc):\n        self._stop.set()\n        self._thread.join()\n        d, h = self.device_used(), self._host()             # one last reading, after the work\n        self.device = None if d is None else max(self.dmax, d) - self.d0\n        self.host = None if h is None else max(self.hmax, h) - self.h0\n\n\ndef _mb(v):\n    return \"?\" if v is None else f\"{v:,.0f}\"\n\n\nSLOW = 20.0      # seconds: a call slower than this is not repeated, and that contender's larger batches skipped\n\n\ndef bench(n: int, d: int, batches, config_2d: dict | None = None, full: bool = False, tune: bool = False,\n          lean: bool = False) -> None:\n    dev = torch.device(\"cuda\")\n    rng = np.random.default_rng(0)\n    xb = rng.standard_normal((n, d), dtype=np.float32)\n    xq = rng.standard_normal((max(batches), d), dtype=np.float32)\n    x, q = torch.from_numpy(xb).to(dev), torch.from_numpy(xq).to(dev)\n    k = 10\n    print(f\"{torch.cuda.get_device_name()}: {n:,} Gaussian vectors, {d} dimensions, L1, k = {k}; queries per second\", flush=True)\n\n    start = time.perf_counter()\n\n    def elapsed():\n        return f\"{(time.perf_counter() - start) / 60:.1f} min so far\"\n\n    def timed(fn, qb):\n        \"\"\"Queries per second, best of two calls after a warm-up; a slow contender (a warm-up past SLOW s) is\n        timed by its warm-up alone, and flagged so that its larger batches are skipped.\"\"\"\n        t = time.perf_counter()\n        fn(qb)\n        torch.cuda.synchronize()\n        warm = time.perf_counter() - t\n        if warm > SLOW:\n            return len(qb) / warm, True\n        best = warm\n        for _ in range(2):\n            t = time.perf_counter()\n            fn(qb)\n            torch.cuda.synchronize()\n            best = min(best, time.perf_counter() - t)\n        return len(qb) / best, False\n\n    def cdist(qb):\n        out = []\n        for s in range(0, len(qb), 64):                            # cdist's broadcast exceeds memory beyond\n            out.append(torch.topk(torch.cdist(qb[s:s + 64], x, p=1), k, largest=False).indices)\n        return torch.cat(out)\n    xt = x.T.contiguous()                                          # the 2D-tile kernel's layout, made once\n    # the 2D kernel's configurations timed one by one, each printed as it is measured (Triton's auto-tuner says\n    # nothing until all are done); the 3D kernel at the configuration its auto-tuner chose on the T4\n    best_cfg, best_t = None, float(\"inf\")\n    configs = [{**c.kwargs, \"num_warps\": c.num_warps} for c in CONFIGS_2D] if tune else [config_2d or BEST_2D]\n    if lean:\n        configs, best_cfg = [], BEST_2D                             # the 2D-tile kernel is not run\n    if configs:\n        print(f\"  the 2D-tile kernel on 1,000 queries: {len(configs)} configuration(s), each compiled (about 30 s \"\n              f\"on a T4) and then timed once (seconds per call; a configuration past {SLOW:.0f} s per call is \"\n              f\"dropped)\", flush=True)\n    for i, cfg in enumerate(configs, 1):\n        if best_cfg is not None and DEADLINE.over(f\"configurations {i}-{len(configs)}\"):\n            break\n        try:\n            l1_knn(x, q[:100], k, cfg, xt=xt)                      # compiles, on a small batch: not timed\n            torch.cuda.synchronize()\n            t = time.perf_counter()\n            l1_knn(x, q[:1000], k, cfg, xt=xt)\n            torch.cuda.synchronize()\n            t = time.perf_counter() - t\n            if t > SLOW:\n                print(f\"    [{i}/{len(configs)}] {cfg}: too slow ({t:.1f} s per call); {elapsed()}\", flush=True)\n                continue\n        except Exception as e:  # noqa: BLE001 (a configuration the GPU cannot run: skip it)\n            print(f\"    [{i}/{len(configs)}] {cfg}: failed ({type(e).__name__}); {elapsed()}\", flush=True)\n            continue\n        print(f\"    [{i}/{len(configs)}] {cfg}: {t:.3f}; {elapsed()}\", flush=True)\n        if t < best_t:\n            best_cfg, best_t = cfg, t\n    if not lean and best_cfg is None:                              # never fall back to Triton's silent auto-tuner\n        raise SystemExit(\"  no 2D-tile configuration ran: nothing to benchmark\")\n    if configs:\n        print(f\"  fastest 2D-tile configuration: {best_cfg}\", flush=True)\n    print(\"  compiling and timing each contender (the merge kernel about 30 s to compile on a T4)\", flush=True)\n    best_3d = {\"BLOCK_Q\": 64, \"BLOCK_N\": 128, \"BLOCK_D\": 2, \"SORT\": True, \"num_warps\": 4}\n    merge_alt = {\"BLOCK_Q\": 64, \"BLOCK_N\": 128, \"num_warps\": 8}\n    contenders = {\"siann, in-kernel merge\": lambda qb: l1_knn_merge(xt, qb, k),\n                  \"siann, merge, 64 x 128, 8w\": lambda qb: l1_knn_merge(xt, qb, k, merge_alt),\n                  \"siann, 2D tiles, tuned\": lambda qb: l1_knn(x, qb, k, best_cfg, xt=xt),\n                  \"siann, 3D tiles, tuned\": lambda qb: l1_knn(x, qb, k, best_3d),\n                  \"torch.cdist + topk\": cdist}\n    only_small = {\"torch.cdist + topk\"}                             # ~90 queries/s on a T4: batch 100 only\n    if not full:                                                    # the short default: siann's kernel, FAISS, cuVS\n        for name in (\"siann, 3D tiles, tuned\", \"torch.cdist + topk\"):\n            del contenders[name]\n    if lean:                                                        # one kernel to compile\n        for name in (\"siann, merge, 64 x 128, 8w\", \"siann, 2D tiles, tuned\"):\n            del contenders[name]\n    try:\n        import faiss\n        import faiss.contrib.torch_utils  # lets FAISS take PyTorch tensors\n        res = faiss.StandardGpuResources()\n        contenders[\"FAISS GPU brute force\"] = lambda qb: faiss.knn_gpu(res, qb, x, k, metric=faiss.METRIC_L1)\n    except Exception as e:  # noqa: BLE001 (FAISS's GPU build missing: skip it)\n        print(f\"  (FAISS GPU skipped: {type(e).__name__}: {e})\")\n    try:\n        from cuvs.neighbors import brute_force\n        built = {}\n\n        def cuvs_search(qb):\n            if \"index\" not in built:                                # built at the first call: counted as kept\n                built[\"index\"] = brute_force.build(x, metric=\"l1\")\n            return brute_force.search(built[\"index\"], qb, k)\n        contenders[\"cuVS brute force\"] = cuvs_search\n    except Exception as e:  # noqa: BLE001 (cuVS missing: skip it)\n        print(f\"  (cuVS skipped: {type(e).__name__}: {e})\")\n    peaks = Peaks()\n\n    def settle():\n        gc.collect()\n        torch.cuda.synchronize()\n        torch.cuda.empty_cache()                                    # PyTorch's cached blocks back to the device\n\n    def memory(fn, qb):\n        \"\"\"PyTorch's peak, the device's and the host's during one call, MB above the start.\"\"\"\n        settle()\n        torch.cuda.reset_peak_memory_stats()\n        start = torch.cuda.memory_allocated()\n        with peaks as p:\n            fn(qb)\n            torch.cuda.synchronize()\n        return (torch.cuda.max_memory_allocated() - start) / 2**20, p.device, p.host\n\n    print(\"  queries per second; then memory per call, MB: PyTorch's peak / the device's peak / the host's peak\",\n          flush=True)\n    print(f\"  {len(contenders)} contenders x {len(batches)} batch sizes; '-' where a batch was skipped\",\n          flush=True)\n    print(f\"  {'':32s}\" + \"\".join(f\"{f'batch {b:,}':>20s}\" for b in batches), flush=True)\n    for row, (name, fn) in enumerate(contenders.items(), 1):\n        settle()\n        before = peaks.device_used()\n        print(f\"  [{row}/{len(contenders)}] {name:26s}\", end=\"\", flush=True)\n        mems, slow = [], False\n        for b in batches:\n            if slow or (name in only_small and b > batches[0]) or DEADLINE.over(f\"{name}, batch {b:,}\"):\n                print(f\"{'-':>20s}\", end=\"\", flush=True)\n                mems.append(None)\n                continue\n            try:\n                qps, slow = timed(fn, q[:b])\n                print(f\"{qps:20,.0f}\", end=\"\", flush=True)\n                mems.append(None if slow else memory(fn, q[:b]))\n            except Exception as e:  # noqa: BLE001 (report and go on: one failure should not hide the rest)\n                print(f\"\\n    batch {b:,} failed: {type(e).__name__}: {str(e)[:200]}\", end=\"\")\n                mems.append(None)\n        print(f\"   ({elapsed()})\", flush=True)\n        cells = [\"\" if m is None else f\"{_mb(m[0])} / {_mb(m[1])} / {_mb(m[2])}\" for m in mems]\n        print(f\"  {'      memory':32s}\" + \"\".join(f\"{c:>20s}\" for c in cells), flush=True)\n        settle()\n        after = peaks.device_used()\n        kept = None if before is None else after - before\n        print(f\"  {'      kept on the device':32s}{_mb(kept):>20s} MB\", flush=True)\n    if \"FAISS GPU brute force\" in contenders:\n        _, fi = contenders[\"FAISS GPU brute force\"](q[:100])\n        fi = torch.as_tensor(fi).cpu().numpy()\n        mine = [(\"in-kernel merge\", lambda: l1_knn_merge(xt, q[:100], k))]\n        if not lean:\n            mine.append((\"2D tiles\", lambda: l1_knn(x, q[:100], k, best_cfg, xt=xt)))\n        for name, run in mine:\n            ours = run()\n            same = (np.sort(fi, 1) == np.sort(ours.cpu().numpy(), 1)).mean()\n            print(f\"  {name}: the same neighbors as FAISS's on 100 queries: {same:.4f}\")\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--check\", action=\"store_true\")\n    p.add_argument(\"--all\", action=\"store_true\", help=\"with --check, every configuration of both kernels\")\n    p.add_argument(\"--bench\", action=\"store_true\")\n    p.add_argument(\"--n\", type=int, default=200_000)\n    p.add_argument(\"--d\", type=int, default=128)\n    p.add_argument(\"--batches\", default=\"100,1000\")\n    p.add_argument(\"--full\", action=\"store_true\", help=\"also round two's kernel and torch.cdist\")\n    p.add_argument(\"--config\", help=\"the 2D kernel's configuration (default: the T4's fastest): BLOCK_Q,BLOCK_N,sort|min,warps\")\n    p.add_argument(\"--lean\", action=\"store_true\", help=\"the merge kernel only (fewer compiles), with FAISS and cuVS\")\n    p.add_argument(\"--tune\", action=\"store_true\", help=\"time all 24 configurations of the 2D kernel first (long)\")\n    p.add_argument(\"--budget\", type=float, default=180, help=\"seconds for the whole run; past it, the rest is skipped\")\n    a = p.parse_args()\n    global DEADLINE\n    DEADLINE = Deadline(a.budget)\n    if a.check:\n        check(a.all)\n    if a.bench:\n        cfg = None\n        if a.config:\n            bq, bn, sel, w = a.config.split(\",\")\n            cfg = {\"BLOCK_Q\": int(bq), \"BLOCK_N\": int(bn), \"SORT\": sel == \"sort\", \"num_warps\": int(w)}\n        bench(a.n, a.d, tuple(map(int, a.batches.split(\",\"))), cfg, a.full, a.tune, a.lean)\n\n\nif __name__ == \"__main__\":\n    main()\n"
with open('triton_probe.py', 'w', encoding='utf-8') as f:
    f.write(source)
print(f"wrote triton_probe.py: {len(source.splitlines())} lines")
print(f"step 3 of 4: Write triton_probe.py: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 4 of 4, Benchmark against cuVS", flush=True)

In [ ]:
# Step 4 of 4: Benchmark against cuVS (about 3 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 4 of 4: Benchmark against cuVS, about 3 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!python -u triton_probe.py --bench --budget {max(20, int(270 - (__import__('time').time() - NB_START)))}
print(f"step 4 of 4: Benchmark against cuVS: done in {(__import__('time').time() - _t0) / 60:.1f} min; that was the last step", flush=True)